<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/04_CIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!kaggle datasets download -d pankrzysiu/cifar10-python
!unzip cifar10-python.zip -d ./cifar10

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class KaggleCIFAR10Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        # [준비 단계] 넘파이 데이터 덩어리와 transform을 주머니 안에 미리 쏙 넣어둡니다.
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, index):
        # [핵심: 실시간 가공] dataset[0] 또는 DataLoader가 데이터를 요청할 때 인덱스에 해당하는 이미지 제공합니다.
        image = self.images[index]
        label = self.labels[index]

        # 파이토치의 transform은 보통 'PIL 이미지' 데이터나 텐서를 선호합니다.
        # 그래서 넘파이 배열을 PIL 이미지 형태로 포장합니다.
        image = Image.fromarray(image)

        # 만약 데이터 증강이나 ToTensor()가 있다면 실시간으로 돌려줍니다.
        if self.transform:
            image = self.transform(image)

        # 가공이 끝난 이미지(Tensor)와 정답(숫자 또는 텐서)을 튜플로 반환합니다.
        # 이때 정답 라벨이 원핫 인코딩 형태나 배열형태라면 정수형태로 변환해줍니다.
        if isinstance(label, np.ndarray):
            label = int(label[0]) if label.size == 1 else int(np.argmax(label))
        else:
            label = int(label)

        return image, label

In [ ]:
import pickle
import numpy as np
import os

def unpickle(file):
    with open(file, 'rb') as fo:
        d = pickle.load(fo, encoding='bytes')
    return d

batch_1_path = './cifar10/cifar-10-batches-py/data_batch_1'
batch_1 = unpickle(batch_1_path)

print(type(batch_1))
print(batch_1.keys())

In [ ]:
print(batch_1[b'data'].shape)

In [ ]:
all_images, all_labels = [], []

for i in range(1, 6):
    batch_path = f"./cifar10/cifar-10-batches-py/data_batch_{i}"
    batch_dict = unpickle(batch_path)

    # 이미지와 라벨을 각각 리스트에 모아줍니다.
    all_images.append(batch_dict[b'data'])
    all_labels.extend(batch_dict[b'labels']) # 라벨은 리스트라서 extend로 이어붙입니다.

# 리스트로 모인 분할 배열들을 하나의 거대한 넘파이 통배열로 병합 (50000, 3072)
x_train_raw = np.vstack(all_images)
y_train = np.array(all_labels)
print(x_train_raw.shape)

# 1차원으로 펴진 것을 (채널, 세로, 가로)로 쪼갠 후 (세로, 가로, 채널)로 축 변경
x_train = x_train_raw.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

print(x_train.shape)
print(y_train.shape)
print(y_train[0])

In [ ]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5), # 50% 확률로 좌우 반전
    transforms.RandomCrop(32, padding=4), # 사방으로 4픽셀 패딩 주고 32x32로 랜덤 크롭
    transforms.ToTensor(), # PIL 이미지를 파이토치 텐서[C, H, W]로 변환 및 0~1 정규화
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # -1~1 정규화
])

# 우리가 만든 커스텀 주머니에 '진짜 데이터'와 '가공 공장' 채워넣기
train_dataset = KaggleCIFAR10Dataset(images=x_train, labels=y_train, transform=train_transform)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(len(train_dataset))
print(len(train_loader))

In [ ]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(labels[0].item())

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 3
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i+1) % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {i+1} | Loss: {running_loss / 100:.4f}")
            running_loss = 0.0

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

def imshow(img):
    # 역정규화(Normalized [-1, 1] -> [0, 1])
    img = img / 2 + 0.5

    npimg = img.numpy()

    # [채널, 세로, 가로] -> [세로, 가로, 채널]로 축 전환.
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

dataiter = iter(train_loader)
images, labels = next(dataiter)

img_grid = torchvision.utils.make_grid(images[:8], nrow=4)

plt.figure(figsize=(10, 5))
imshow(img_grid)
plt.show()
print([classes[labels[j]] for j in range(8)])